In [4]:
import pandas as pd
import os

# 1. Đọc dữ liệu
df = pd.read_csv("D:\Downloads\DATA_THTN\moho_data\products_ready_to_db.csv", encoding='utf-8-sig')

sql_statements = []
sql_statements.append("SET FOREIGN_KEY_CHECKS = 0;")
sql_statements.append("USE sale_tmdt;")

# 2. SQL CHO CATEGORY
categories = df[['category_name', 'category_slug']].drop_duplicates()
cat_id_map = {}
for i, (idx, row) in enumerate(categories.iterrows(), 1):
    c_name = row['category_name']
    c_slug = row['category_slug']
    cat_id_map[c_name] = i
    # Dùng ON DUPLICATE KEY để tránh lỗi trùng Category
    sql_statements.append(f"INSERT INTO app_category (id, name, slug, is_sub) VALUES ({i}, '{c_name}', '{c_slug}', 0) ON DUPLICATE KEY UPDATE name=name;")

# 3. SQL CHO PRODUCT
for p_id, (index, row) in enumerate(df.iterrows(), 1):
    name_esc = str(row['name']).replace("'", "''")
    desc_esc = str(row['description']).replace("'", "''")
    material_esc = str(row['material']).replace("'", "''") if pd.notna(row['material']) else ""
    size_esc = str(row['size']).replace("'", "''") if pd.notna(row['size']) else ""
    
    db_main_img_path = f"main/{row['image_main']}"
    sale_price = row['sale_price'] if pd.notna(row['sale_price']) else "NULL"
    
    # Dùng REPLACE INTO hoặc INSERT IGNORE để nếu chạy lại nhiều lần không bị lỗi ID
    sql_prod = (f"REPLACE INTO app_product (id, name, price, sale_price, digital, image, description, material, size, origin, quality) "
                f"VALUES ({p_id}, '{name_esc}', {row['price']}, {sale_price}, 0, '{db_main_img_path}', '{desc_esc}', '{material_esc}', '{size_esc}', 'Việt Nam', 'Loại 1');")
    sql_statements.append(sql_prod)
    
    # 4. SQL CHO BẢNG TRUNG GIAN (Dùng INSERT IGNORE để tránh lỗi Duplicate entry 1-1)
    c_id = cat_id_map.get(row['category_name'])
    sql_statements.append(f"INSERT IGNORE INTO app_product_category (product_id, category_id) VALUES ({p_id}, {c_id});")
    
    # 5. SQL CHO ẢNH PHỤ
    if pd.notna(row['images_sub']):
        sub_images = row['images_sub'].split(',')
        for img_rel_path in sub_images:
            db_sub_path = f"multiple/{img_rel_path.strip()}"
            sql_statements.append(f"INSERT IGNORE INTO app_productimage (product_id, image, date_added) VALUES ({p_id}, '{db_sub_path}', NOW());")

sql_statements.append("SET FOREIGN_KEY_CHECKS = 1;")

# Xuất ra file SQL
with open(r'D:\Big_project_2025\WebShop\final_insert_to_mysql.sql', 'w', encoding='utf-8') as f:
    f.write("\n".join(sql_statements))

print("--- ĐÃ CẬP NHẬT FILE SQL VỚI LỆNH TRÁNH TRÙNG LẶP ---")

--- ĐÃ CẬP NHẬT FILE SQL VỚI LỆNH TRÁNH TRÙNG LẶP ---
